In [10]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# =========================
# 1. Load Data
# =========================
df = pd.read_csv("raw_loan_dataset.csv")

print(df.head())
print(df.info())
print(df.isnull().sum())

# =========================
# 2. Clean currency columns
# =========================
df["Income"] = df["Income"].replace('[\$,]', '', regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace('[\$,]', '', regex=True).astype(float)

# =========================
# 3. Fix category errors
# =========================
df["HasCollateral"] = df["HasCollateral"].str.lower().replace({
    "yes":"Yes", "no":"No", "y":"Yes", "n":"No"
})

df["PreviousDefaults"] = df["PreviousDefaults"].str.lower().replace({
    "yes":"Yes", "no":"No", "y":"Yes", "n":"No"
})

df["Approved"] = df["Approved"].str.lower().replace({
    "approved":"Yes", "rejected":"No", "yes":"Yes", "no":"No"
})

print(df["Approved"].value_counts())

# =========================
# 4. Missing values
# =========================
num_cols = df.select_dtypes(include="number").columns
cat_cols = df.select_dtypes(include="object").columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

print(df.isnull().sum())

# =========================
# 5. Remove duplicates
# =========================
print("Before:", df.shape)
df = df.drop_duplicates()
print("After:", df.shape)

# =========================
# 6. Outlier handling (IQR capping)
# =========================
cols = ["Income", "CreditScore", "LoanAmount", "EmploymentYears"]

for col in cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

# =========================
# 7. Encode labels
# =========================
df["Approved"] = df["Approved"].map({"Yes":1, "No":0})
df["HasCollateral"] = df["HasCollateral"].map({"Yes":1, "No":0})
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes":1, "No":0})

print(df["Approved"].value_counts())

# =========================
# 8. Feature engineering
# =========================
df["DebtToIncome"] = df["LoanAmount"] / df["Income"]
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

# =========================
# 9. Scaling (MinMaxScaler)
# =========================
scaler = MinMaxScaler()

scale_cols = ["Income", "CreditScore", "LoanAmount", "EmploymentYears"]

df[scale_cols] = scaler.fit_transform(df[scale_cols])

# =========================
# 10. Save clean dataset
# =========================
df.to_csv("clean_loan_dataset.csv", index=False)

print(df.head())
print(df.isnull().sum())

    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasColla